In [ ]:
import pandas as pd
from scipy import stats

In [2]:
zakaznici=pd.read_csv("obj.CSV")

zakaznici.head()

,id_objednavky,id_zakaznik,datum,čas,cena_celkem,zpusob_platby,store_typ,id_pobocky,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,1,275,25.09.2021,15:35,2900,prevod,online,0,NaN,NaN,NaN,NaN,NaN
1,2,586,27.07.2022,13:44,5000,karta,online,0,NaN,NaN,NaN,NaN,NaN
2,3,588,01.10.2020,18:19,9200,karta,online,0,NaN,NaN,NaN,NaN,NaN
3,4,1077,04.07.2025,9:17,3900,prevod,online,0,NaN,NaN,NaN,NaN,NaN
4,5,1580,30.10.2020,15:12,3300,karta,online,0,NaN,NaN,NaN,NaN,NaN


**Agregace dat: počet nákupů a průměrná objednávka na zákazníka**


In [34]:
case = zakaznici.groupby('id_zakaznik').agg(
    pocet_nakupu=('id_objednavky', 'count'),
    prumerna_objednavka=('cena_celkem', 'mean')
).reset_index()
case.describe()


,id_zakaznik,pocet_nakupu,prumerna_objednavka
count,2984.000000,2984.000000,2984.000000
mean,1492.500000,2.947386,5112.111892
std,861.550927,1.763292,3065.199695
min,1.000000,1.000000,0.000000
25%,746.750000,1.000000,3065.625000
50%,1492.500000,3.000000,4535.416667
75%,2238.250000,4.000000,6450.000000
max,2984.000000,13.000000,28650.000000


**Segmentace zákazníků na základě percentilů**

In [35]:
case["typ_zakaznik"] = pd.cut(
    case["pocet_nakupu"], 
    bins=[0, 1, 3, 13], 
    labels=["jednorazovy", "prilezitostny", "verny"]
    )

case.head()

,id_zakaznik,pocet_nakupu,prumerna_objednavka,typ_zakaznik
0,1,1,3000.000000,jednorazovy
1,2,2,4750.000000,prilezitostny
2,3,3,5466.666667,prilezitostny
3,4,5,6930.000000,verny
4,5,5,6750.000000,verny


**ANOVA: existuje statisticky významný rozdíl mezi segmenty?**

In [36]:
one = case[case["typ_zakaznik"] == "jednorazovy"]["prumerna_objednavka"]
two = case[case["typ_zakaznik"] == "prilezitostny"]["prumerna_objednavka"]
three = case[case["typ_zakaznik"] == "verny"]["prumerna_objednavka"]

stats.f_oneway(one, two, three)

F_onewayResult(statistic=np.float64(15.051626167384596), pvalue=np.float64(3.132903344399584e-07))

**Závěr ANOVA**

P-hodnota: 3.13e-07 (prakticky nulová)

Rozdíl v průměrné hodnotě objednávky mezi zákaznickými segmenty 
je statisticky velmi významný. Pro konkrétní hodnoty jednotlivých 
segmentů viz následující analýza.

**Průměrná hodnota objednávky podle segmentu**

In [37]:
case.groupby("typ_zakaznik")["prumerna_objednavka"].mean()

typ_zakaznik
jednorazovy      5585.362694
prilezitostny    5072.530864
verny            4777.942969
Name: prumerna_objednavka, dtype: float64

**Závěr**

Ačkoliv byl potvrzen statisticky významný rozdíl mezi segmenty, průměrná hodnota objednávky je paradoxně nejvyšší u jednorázových zákazníků (5 585 Kč) a nejnižší u věrných (4 778 Kč). 

Toto zjištění otevírá další otázky - klíčovým dalším krokem by byla například analýza retention rate a celková hodnota zákazníka v čase (LTV). 

